In [1]:
import itertools
import spot


def extract_atomic_propositions(*formulas: str) -> list[str]:
    """Extract all unique atomic propositions (APs) across given LTL formulas."""
    aps = set()
    for f in formulas:
        parsed = spot.formula(f)
        for ap in spot.atomic_prop_collect(parsed):
            aps.add(ap.ap_name())
    return sorted(list(aps))


def generate_all_traces(aps: list[str], k: int) -> list[tuple[str, ...]]:
    """Generate all 2^(|AP| * k) possible execution traces of length k."""
    if not aps or k <= 0:
        return [()]

    step_valuations = []
    for truth_values in itertools.product([True, False], repeat=len(aps)):
        literals = [ap if is_true else f"!{ap}" for ap, is_true in zip(aps, truth_values)]
        step_valuations.append(" & ".join(literals))

    return list(itertools.product(step_valuations, repeat=k))


def get_satisfying_traces(formula_str: str, aps: list[str], k: int) -> set[tuple[str, ...]]:
    """Finds all length-k traces that satisfy the formula under LTLf semantics."""
    # Convert LTLf formula string -> LTL formula -> Automaton graph (twa_graph)
    aut = spot.translate(spot.from_ltlf(formula_str))
    dict_ptr = aut.get_dict()
    
    all_traces = generate_all_traces(aps, k)
    satisfying_traces = set()

    for trace in all_traces:
        if trace:
            prefix = " ; ".join(f"({step}) & alive" for step in trace)
            word_str = f"{prefix} ; cycle{{!alive}}"
        else:
            word_str = "cycle{!alive}"

        word = spot.parse_word(word_str, dict_ptr)
        word_aut = word.as_automaton()

        if not spot.product(aut, word_aut).is_empty():
            satisfying_traces.add(trace)

    return satisfying_traces


def ltl_jaccard_metrics(f1: str, f2: str, k: int, aps: list[str] = None) -> tuple[float, float]:
    """Computes (Jaccard Similarity, Jaccard Distance) between two LTL formulas over horizon k."""
    if aps is None:
        aps = extract_atomic_propositions(f1, f2)

    t1 = get_satisfying_traces(f1, aps, k)
    t2 = get_satisfying_traces(f2, aps, k)

    intersection_size = len(t1.intersection(t2))
    union_size = len(t1.union(t2))

    if union_size == 0:
        similarity = 1.0
    else:
        similarity = intersection_size / union_size

    distance = 1.0 - similarity
    return similarity, distance


def compute_pairwise_similarity_matrix(formulas: list[str], k: int) -> tuple[list[list[float]], list[str]]:
    """Generates an NxN pairwise similarity matrix for candidate responses."""
    aps = extract_atomic_propositions(*formulas)
    n = len(formulas)
    matrix = [[1.0] * n for _ in range(n)]

    trace_sets = [get_satisfying_traces(f, aps, k) for f in formulas]

    for i in range(n):
        for j in range(i + 1, n):
            t1, t2 = trace_sets[i], trace_sets[j]
            union_len = len(t1.union(t2))
            sim = 1.0 if union_len == 0 else len(t1.intersection(t2)) / union_len
            matrix[i][j] = sim
            matrix[j][i] = sim

    return matrix, aps


# Example Usage
if __name__ == "__main__":
    f1 = "G(a -> F b)"
    f2 = "!(F(a & G(!b)))"  # Semantically equivalent to f1
    f3 = "F a"              # Semantically distinct

    horizon = 5

    sim_12, dist_12 = ltl_jaccard_metrics(f1, f2, k=horizon)
    print(f"Jaccard Similarity (f1 vs f2): {sim_12:.4f}")
    print(f"Jaccard Distance   (f1 vs f2): {dist_12:.4f}\n")

    sim_13, dist_13 = ltl_jaccard_metrics(f1, f3, k=horizon)
    print(f"Jaccard Similarity (f1 vs f3): {sim_13:.4f}")
    print(f"Jaccard Distance   (f1 vs f3): {dist_13:.4f}\n")

    responses = [f1, f2, f3]
    matrix, aps = compute_pairwise_similarity_matrix(responses, k=horizon)
    
    print(f"Atomic Propositions evaluated: {aps}")
    print("Similarity Matrix s(i, j):")
    for row in matrix:
        print([round(val, 3) for val in row])

Jaccard Similarity (f1 vs f2): 1.0000
Jaccard Distance   (f1 vs f2): 0.0000

Jaccard Similarity (f1 vs f3): 0.6357
Jaccard Distance   (f1 vs f3): 0.3643

Atomic Propositions evaluated: ['a', 'b']
Similarity Matrix s(i, j):
[1.0, 1.0, 0.636]
[1.0, 1.0, 0.636]
[0.636, 0.636, 1.0]


In [2]:
import numpy as np

def compute_ambiguity(probabilities: list[float], similarity_matrix: list[list[float]]) -> float:
    """
    Computes the normalized ambiguity score A for a set of candidate LTL formulas.
    
    Args:
        probabilities: Response probability distribution p (sums to 1.0).
        similarity_matrix: NxN matrix s(i, j) of pairwise Jaccard similarities.
        
    Returns:
        Ambiguity score A in [0, 1].
    """
    p = np.array(probabilities, dtype=float)
    s = np.array(similarity_matrix, dtype=float)
    
    # 1. Entropy H (using natural log)
    p_nz = p[p > 0]
    H = -np.sum(p_nz * np.log(p_nz))
    
    # 2. Uncertainty scaling factor: (1 - e^-H)
    uncertainty_term = 1.0 - np.exp(-H)
    
    # 3. Normalized spread S_norm
    numerator = np.sum(np.outer(p, p) * (1.0 - s))
    denominator = 1.0 - np.sum(p**2)
    
    # Edge case: If all probability mass is concentrated on a single response
    S_norm = 0.0 if denominator == 0 else numerator / denominator
    
    return float(uncertainty_term * S_norm)

# Example: Candidate responses f1, f2, f3 with empirical probabilities
probabilities = [0.5, 0.3, 0.2]  # f1 (50%), f2 (30%), f3 (20%)

# Matrix generated from Spot
similarity_matrix = [
    [1.0,   1.0,   0.636],
    [1.0,   1.0,   0.636],
    [0.636, 0.636, 1.0  ]
]

ambiguity_score = compute_ambiguity(probabilities, similarity_matrix)
print(f"Final Ambiguity Score (A): {ambiguity_score:.4f}")

Final Ambiguity Score (A): 0.1208


In [ ]:
import sqlite3
import json
import pandas as pd

def export_flattened_results(db_path="experiment_results.db", output_csv="flattened_results.csv"):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    
    # Grab all completed experiment runs
    cursor = conn.execute("SELECT * FROM experiments_07 WHERE status = 'COMPLETED'")
    rows = cursor.fetchall()
    conn.close()

    all_retakes = []

    for row in rows:
        if row['result_output']:
            # Decode the JSON array containing the 20 retakes
            retakes_data = json.loads(row['result_output'])
            all_retakes.extend(retakes_data)

    # Convert to a DataFrame and export
    df = pd.DataFrame(all_retakes)
    df.to_csv(output_csv, index=False)
    print(f"✅ Successfully exported {len(df)} total retake results to '{output_csv}'.")
    return df

# Run the export
df = export_flattened_results()
print(df.head())

✅ Successfully exported 8775200 total retake results to 'flattened_results.csv'.
  PROMPTTYPE  experiment_index                 Requirement  \
0      BASIC                 2  keep apple until drop pear   
1      BASIC                 2  keep apple until drop pear   
2      BASIC                 2  keep apple until drop pear   
3      BASIC                 2  keep apple until drop pear   
4      BASIC                 2  keep apple until drop pear   

             Ground Truth           Atomic Proposition  \
0  keep_apple U drop_pear  ['drop_pear', 'keep_apple']   
1  keep_apple U drop_pear  ['drop_pear', 'keep_apple']   
2  keep_apple U drop_pear  ['drop_pear', 'keep_apple']   
3  keep_apple U drop_pear  ['drop_pear', 'keep_apple']   
4  keep_apple U drop_pear  ['drop_pear', 'keep_apple']   

        Original Response                Response Equivalent  
0  keep_apple U drop_pear  keep_apple U drop_pear       True  
1  keep_apple U drop_pear  keep_apple U drop_pear       True  
2  keep_

In [ ]:
df

,PROMPTTYPE,experiment_index,Requirement,Ground Truth,Atomic Proposition,Original Response,Response,Equivalent
0,BASIC,2,keep apple until drop pear,keep_apple U drop_pear,"['drop_pear', 'keep_apple']",keep_apple U drop_pear,keep_apple U drop_pear,True
1,BASIC,2,keep apple until drop pear,keep_apple U drop_pear,"['drop_pear', 'keep_apple']",keep_apple U drop_pear,keep_apple U drop_pear,True
2,BASIC,2,keep apple until drop pear,keep_apple U drop_pear,"['drop_pear', 'keep_apple']",keep_apple U drop_pear,keep_apple U drop_pear,True
3,BASIC,2,keep apple until drop pear,keep_apple U drop_pear,"['drop_pear', 'keep_apple']",keep_apple U drop_pear,keep_apple U drop_pear,True
4,BASIC,2,keep apple until drop pear,keep_apple U drop_pear,"['drop_pear', 'keep_apple']",keep_apple U drop_pear,keep_apple U drop_pear,True
...,...,...,...,...,...,...,...,...
8775195,ADARULE,14337,For each time instant in the future it should ...,prop_3 & G(prop_1 -> prop_2),"['prop_1', 'prop_2', 'prop_3']",So the final LTL translation is: G(prop_1 -> (...,G(prop_1 -> (prop_2 & prop_3)),False
8775196,ADARULE,14337,For each time instant in the future it should ...,prop_3 & G(prop_1 -> prop_2),"['prop_1', 'prop_2', 'prop_3']",So the final LTL translation is: G(prop_1 -> (...,G(prop_1 -> (prop_2 & prop_3)),False
8775197,ADARULE,14337,For each time instant in the future it should ...,prop_3 & G(prop_1 -> prop_2),"['prop_1', 'prop_2', 'prop_3']",So the final LTL translation is: G(prop_1 -> (...,G(prop_1 -> (F(prop_2) & prop_3)),False
8775198,ADARULE,14337,For each time instant in the future it should ...,prop_3 & G(prop_1 -> prop_2),"['prop_1', 'prop_2', 'prop_3']",So the final LTL translation is: G((prop_1) ->...,G((prop_1) -> (prop_2) & (prop_3)),False


In [ ]:
1